# GARCH Modelling

In [321]:
%load_ext pretty_jupyter


# Introduction

When working with financial time series in Python, we often have access to high-frequency observations. For example, it is common to analyze daily observations, but depending on the asset or market, even higher frequencies—such as hourly, minute, second, or millisecond—are available.

For simplicity, we will restrict our analysis to **daily observations**. These could vary based on the asset type:

- For some assets, daily data may be available for all 7 days of the week.
- For others, such as stocks traded on exchanges, daily data is often limited to weekdays (work days), typically resulting in 5 weekly observations.

# Packages Used for GARCH Modeling in Python

In Python, several libraries provide robust tools for estimating volatility models, including univariate and multivariate GARCH. For instance:

1. `arch`: Provides a comprehensive suite of tools for estimating univariate GARCH models, written specifically for volatility modeling.
2. `statsmodels`: Useful for foundational econometric and time-series analysis.
3. `yfinance` or `pandas-datareader`: Enables easy access to financial data from sources like Yahoo Finance or FRED (Federal Reserve Economic Data).

To begin, ensure that you have installed these packages in your Python environment. You can install them using `pip`:

In [29]:
# Install necessary packages
!pip install arch yfinance statsmodels

Also, ensure that you load the required libraries

In [31]:
import os
import pandas as pd
from arch import arch_model  # For GARCH modeling
import yfinance as yf  # For downloading financial data
from statsmodels.tsa.stattools import adfuller  # For stationarity tests
import matplotlib.pyplot as plt  # For visualizations

Next we set our working directory

In [ ]:
# Set working directory (replace with your directory path if needed)
# os.chdir("YOUR/COMPLETE/DIRECTORY/PATH")

# Data upload

Here, we will use a convenient data retrieval function provided by the yfinance library in Python to retrieve financial data. This library allows us to download stock data from Yahoo Finance effortlessly. For instance, you can use it to retrieve data for stock tickers such as IBM, Google, or market indices like the S&P 500.

If you're unsure about the ticker symbol for a specific stock, you can easily search online for a list of ticker symbols.  The default source is [Yahoo Finance](https://finance.yahoo.com/). Below, you'll find an example demonstrating how to use the yfinance library to fetch stock data. Note that occasionally, you might encounter connection issues or errors when fetching data. In such cases, simply retry after a few seconds, and it should work fine. In this example, we will use the `yfinance` library to provide a simple and efficient way to retrieve stock market data from Yahoo Finance.

In [35]:
import yfinance as yf
import pandas as pd

# Define the start and end dates for the data retrieval
start_date = "2007-01-03"
end_date = "2018-04-30"

# Retrieve data for multiple tickers (e.g., GSPC, IBM, GOOG, BP)
tickers = ["^GSPC", "IBM", "GOOG", "BP"]

# Download data for all tickers
data = yf.download(tickers, start=start_date, end=end_date)

# Display the structure of the data
print(data.info())  # Overview of the data
print(data.head())  # First few rows of the dataset

[*********************100%***********************]  4 of 4 completed

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 2850 entries, 2007-01-03 to 2018-04-27
Data columns (total 24 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   (Adj Close, BP)     2850 non-null   float64
 1   (Adj Close, GOOG)   2850 non-null   float64
 2   (Adj Close, IBM)    2850 non-null   float64
 3   (Adj Close, ^GSPC)  2850 non-null   float64
 4   (Close, BP)         2850 non-null   float64
 5   (Close, GOOG)       2850 non-null   float64
 6   (Close, IBM)        2850 non-null   float64
 7   (Close, ^GSPC)      2850 non-null   float64
 8   (High, BP)          2850 non-null   float64
 9   (High, GOOG)        2850 non-null   float64
 10  (High, IBM)         2850 non-null   float64
 11  (High, ^GSPC)       2850 non-null   float64
 12  (Low, BP)           2850 non-null   float64
 13  (Low, GOOG)         2850 non-null   float64
 14  (Low, IBM)          2850 non-null   float64
 15  (Low, ^GSPC)        2850 non-null   f

In `yfinance`, the data for multiple tickers is returned in a multi-index DataFrame. You can access individual stocks as follows:

In [37]:
# Access data for a specific ticker (e.g., IBM)
ibm_data = data['Adj Close']['IBM']  # Adjusted Close prices for IBM
print(ibm_data.head())

# Access structure
print(ibm_data.describe())  # Summary statistics

Date
2007-01-03    52.171749
2007-01-04    52.729584
2007-01-05    52.252205
2007-01-08    53.046024
2007-01-09    53.673576
Name: IBM, dtype: float64
count    2850.000000
mean       91.502915
std        22.965805
min        39.702785
25%        70.363295
50%        98.966640
75%       111.115341
max       128.909210
Name: IBM, dtype: float64


To inspect the structure and details of the dataset:

In [51]:
# Display the structure of the IBM dataset
print(ibm_data.info())
print(ibm_data.head())

<class 'pandas.core.series.Series'>
DatetimeIndex: 2850 entries, 2007-01-03 to 2018-04-27
Series name: IBM
Non-Null Count  Dtype  
--------------  -----  
2850 non-null   float64
dtypes: float64(1)
memory usage: 44.5 KB
None
Date
2007-01-03    52.171749
2007-01-04    52.729584
2007-01-05    52.252205
2007-01-08    53.046024
2007-01-09    53.673576
Name: IBM, dtype: float64


You can see that this object contains a range of daily observations (Open, High, Close, Volume and Adjusted share price). We also learn that the IBM data is stored as a `pandas.Series` within a larger `pandas.DataFrame`. This is beneficial because `Series` objects come with a lot of built-in methods that are handy for time-series analysis, such as `.resample()`, `.asfreq()`, and `.rolling()`, among others. These methods allow for easy manipulation and analysis of time-series data. Moreover, the` DatetimeIndex` indicates that the data is indexed by dates, making it suitable for time-series analysis. The index spans from **2007-01-03** to **2018-04-27**, corresponding to the data range retrieved from Yahoo Finance.  

Let's plot the adjusted close prices of a stock using lets_plot with a line chart. 

In [56]:
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()

# Create your date range and adjusted close prices
dates = pd.date_range(start="2007-01-03", end="2018-04-30", freq='B')  # Business days
prices = np.random.normal(loc=150, scale=10, size=len(dates))  # Simulated prices

# Creating the DataFrame
data_lets_plot_df = pd.DataFrame({'date': dates, 'Adjusted Close Price': prices})

# Plotting using lets_plot
plot = (ggplot(data_lets_plot_df, aes(x='date', y='Adjusted Close Price')) +
        geom_line(color='darkblue', size=1.5) +
        ggtitle("IBM Stock Adjusted Close Price (2007-2018)") +
        xlab("Date") +
        ylab("Adjusted Close Price (USD)") +
        theme_minimal() +
        ggsize(1200, 600))  # Set the size here using ggsize(width, height)

# Display the plot
plot

Here's a brief example of using a rolling window to calculate a simple moving average, which can help smooth out short-term fluctuations and highlight longer-term trends in the stock price:

In [60]:
import pandas as pd
import numpy as np
from lets_plot import *
LetsPlot.setup_html()

# Create your date range and adjusted close prices
dates = pd.date_range(start="2007-01-03", end="2018-04-30", freq='B')  # Business days
prices = np.random.normal(loc=150, scale=10, size=len(dates))  # Simulated prices

# Creating the DataFrame
data_lets_plot_df = pd.DataFrame({'date': dates, 'Adjusted Close Price': prices})

# Calculate the 30-day simple moving average
data_lets_plot_df['30-day SMA'] = data_lets_plot_df['Adjusted Close Price'].rolling(window=30).mean()

# Plotting using lets_plot
plot = (ggplot(data_lets_plot_df, aes(x='date')) +
        geom_line(aes(y='Adjusted Close Price'), color='darkblue', size=1.5, legend='Adjusted Close') +
        geom_line(aes(y='30-day SMA'), color='orange', size=1.5, legend='30-day SMA') +
        ggtitle("IBM Stock Adjusted Close Price and 30-day SMA (2007-2018)") +
        xlab("Date") +
        ylab("Price (USD)") +
        theme_minimal() +
        ggsize(1200, 600))  # Set the size here using ggsize(width, height)

# Display the plot
plot

When we are estimating volatility models we work with returns. There is a function in Python that transforms the data to returns. The `pandas` method `.pct_change()` is commonly used to calculate returns from series of prices. Here's how you can do it:

1. **Fetch or use existing data**: If you already have the price data as a DataFrame (similar to the earlier example with `data`), you can proceed directly. Otherwise, you might need to fetch data using, for example, `yfinance`.

2. **Calculate Returns**: Use the `.pct_change()` function to calculate daily returns from the adjusted close prices.

3. **Prepare the DataFrame for Multivariate Analysis*: Combine the returns into a single DataFrame.

In [63]:
import pandas as pd
import yfinance as yf

# Fetching data (if not already loaded)
tickers = ["IBM", "BP", "GOOG"]
data = yf.download(tickers, start="2007-01-03", end="2018-04-30")['Adj Close']

# Calculating daily returns
daily_returns = data.pct_change()

# Rename the columns
daily_returns.columns = ['rIBM', 'rBP', 'rGOOG']

# Optionally, you can calculate weekly returns as well
weekly_returns = data.resample('W').ffill().pct_change()

# Display the daily returns DataFrame
print(daily_returns.head())


[*********************100%***********************]  3 of 3 completed

                rIBM       rBP     rGOOG
Date                                    
2007-01-03       NaN       NaN       NaN
2007-01-04 -0.013186  0.033512  0.010692
2007-01-05 -0.002150  0.008132 -0.009053
2007-01-08 -0.010467 -0.007410  0.015192
2007-01-09 -0.028776  0.003970  0.011830


Here, the `yfinance.download()` function is used to fetch historical stock data from Yahoo Finance for IBM, BP, and Google over a specified date range. The `.pct_change()` method computes the percentage change from the previous row by default, which is commonly used to compute returns in financial data analysis. Then the `resample('W')` method is used with `.ffill()` to forward-fill missing data, ensuring that the price data is aligned weekly before calculating percentage changes for weekly returns.

#  Univariate GARCH Model

In Python, the equivalent for conducting GARCH modeling, similar to using the `rugarch` package in R, is the `arch` package. The `arch` package, developed by Kevin Sheppard, provides comprehensive tools to model and estimate the volatility of financial time series, including various forms of GARCH models. It's widely used for financial econometrics tasks such as volatility modeling in Python. 

Using the `arch` package, you can specify different types of GARCH models and change their configurations, similar to the way you would set parameters in R's `rugarch` package with `ugarchspec()`. The `arch` package offers a flexible interface to specify various components of the GARCH model, including the mean model and the conditional distribution.

## Basic Model Specification in Python

To create a basic GARCH(1,1) model specification in Python:

In [76]:
import yfinance as yf
from arch import arch_model

# Fetching data
data = yf.download('AAPL', start='2010-01-01', end='2020-01-01')
prices = data['Adj Close']

# Calculating returns
returns = 100 * prices.pct_change().dropna()  # Convert to percentage and remove NA values

# Specifying a basic GARCH(1,1) model with a constant mean
model_spec = arch_model(returns, mean='Constant', vol='Garch', p=1, q=1, dist='normal')
model_fit = model_spec.fit(disp='off')  # Fit the model without displaying the fit progress

# Printing model summary
print(model_fit.summary())

[*********************100%***********************]  1 of 1 completed

                     Constant Mean - GARCH Model Results                      
Dep. Variable:                   AAPL   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:               -4687.92
Distribution:                  Normal   AIC:                           9383.83
Method:            Maximum Likelihood   BIC:                           9407.15
                                        No. Observations:                 2515
Date:                Mon, Jan 20 2025   Df Residuals:                     2514
Time:                        00:41:30   Df Model:                            1
                               Mean Model                               
                 coef    std err          t      P>|t|  95.0% Conf. Int.
------------------------------------------------------------------------
mu             0.1792  3.248e-02      5.517  3.442e-08 [  0.116,  0.24

In this case, the `pct_change()` method calculates the percentage change from one period to the next, multiplied by 100 to convert it to a percentage format. Then, the `arch_model` function is used to specify a GARCH(1,1) model. The parameters include setting the mean model to 'Constant' (implying a constant mean in the return equation), the volatility model to 'Garch', and the distribution of the residuals to 'normal'. The `fit()` method is called with `disp='off'` to fit the model to the returns data without displaying the iterative fitting process. 

> It is important to note that the the `arch` package supports several mean models, but the most commonly used ones are:
     - `Constant` (just an intercept)
     - `Zero` (no mean, the mean is assumed to be zero)
     - `AR` (autoregressive model of order X, specified by lags).

The key aspects of the model setup include the specification for the Mean Model, which in this case is a constant mean model, and the Volatility Model, specified as an sGARCH(1,1) — a standard GARCH(1,1) model. This setup provides a basic yet powerful framework for analyzing financial time series volatility. To get details on all the possible specifications and how to change them it is best to consult the [documentation](https://arch.readthedocs.io/en/latest/) of the `arch` package. 

Let’s say you want to change the mean model from a constant mean model to an ARMA(1,0), which is essentially an autoregressive model of order 1 (AR(1)). 

#### What will change now?

[//]: # (-.- .tabset)

##### Specify the Model

Change the `mean` parameter from `'Constant'` to `'AR'`, and set the `lags` parameter to `[1]` to indicate an AR(1) model. This tells the model to include one lag of the dependent variable (returns) in the mean equation.


In [89]:
import yfinance as yf
from arch import arch_model

# Fetching data
data = yf.download('AAPL', start='2010-01-01', end='2020-01-01')
prices = data['Adj Close']

# Calculating returns
returns = 100 * prices.pct_change().dropna()  # Convert to percentage and remove NA values

# Specifying a GARCH(1,1) model with an AR(1) mean model
model_spec = arch_model(returns, mean='AR', lags=[1], vol='Garch', p=1, q=1, dist='normal')
model_fit = model_spec.fit(disp='off')  # Fit the model without displaying the fit progress

# Printing model summary
print(model_fit.summary())


[*********************100%***********************]  1 of 1 completed


                           AR - GARCH Model Results                           
Dep. Variable:                   AAPL   R-squared:                      -0.002
Mean Model:                        AR   Adj. R-squared:                 -0.002
Vol Model:                      GARCH   Log-Likelihood:               -4685.95
Distribution:                  Normal   AIC:                           9381.89
Method:            Maximum Likelihood   BIC:                           9411.04
                                        No. Observations:                 2514
Date:                Mon, Jan 20 2025   Df Residuals:                     2512
Time:                        00:52:54   Df Model:                            2
                                  Mean Model                                 
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
Const          0.1766  3.327e-02      5.308  1.110e-07 

###### What can we learn? 
By specifying `lags=[1]`, you instruct the `arch` package to use the first lag of the returns as the predictor in the mean equation. This models the mean of the returns as being dependent on its previous value, which is typical in time series analysis where past values might influence current values.

The inclusion of an AR term can help capture autocorrelations in the return series, potentially leading to a model that better fits the data if such autocorrelations exist. This is particularly useful if preliminary analysis suggests that returns are autocorrelated, which is often checked using plots like the autocorrelation function (ACF) or partial autocorrelation function (PACF).

This approach effectively transitions your model from focusing solely on volatility modeling with a static mean to incorporating dynamics in the mean based on historical data. This can be especially beneficial for forecasting and risk management applications where understanding the relationships in the data is crucial.

[//]: # (-.- .unlisted .unnumbered)

# Model Estimation

Now that we have specified a GARCH model in Python, the next step is to find the best parameters, i.e., we need to estimate the model. This is accomplished using the `fit` method provided by the `arch_model` function from the `arch` library. By fitting the model, we employ maximum likelihood estimation to optimize the model parameters and ensure the best fit to the data. For instance, let's consider the **Exponential GARCH (EGARCH) model**, which is particularly useful for modelling asymmetries in financial time series volatility. This model can capture the leverage effect, where adverse shocks to asset returns increase future volatility more than positive shocks of the same magnitude.

We'll use the `arch` library again because, as mentioned previously, it supports various GARCH model forms, including EGARCH. Here's how you can set up and estimate an EGARCH model:

In [99]:
import yfinance as yf
from arch import arch_model

# Fetching data
data = yf.download('AAPL', start='2010-01-01', end='2020-01-01')
prices = data['Adj Close']

# Calculating returns
returns = 100 * prices.pct_change().dropna()  # Convert to percentage and remove NA values

# Specifying an EGARCH(1,1) model
model = arch_model(returns, mean='Zero', vol='EGARCH', p=1, o=1, q=1, dist='normal')
model_fit = model.fit(disp='off')

# Printing model summary
print(model_fit.summary())


[*********************100%***********************]  1 of 1 completed

                       Zero Mean - EGARCH Model Results                       
Dep. Variable:                   AAPL   R-squared:                       0.000
Mean Model:                 Zero Mean   Adj. R-squared:                  0.000
Vol Model:                     EGARCH   Log-Likelihood:               -4639.55
Distribution:                  Normal   AIC:                           9287.09
Method:            Maximum Likelihood   BIC:                           9310.41
                                        No. Observations:                 2515
Date:                Mon, Jan 20 2025   Df Residuals:                     2515
Time:                        01:06:44   Df Model:                            0
                              Volatility Model                             
                 coef    std err          t      P>|t|     95.0% Conf. Int.
---------------------------------------------------------------------------
omega          0.0893  2.183e-02      4.088  4.356e-05  [4.64

EGARCH models are particularly adept at handling the asymmetric effects of shocks on volatility, unlike standard GARCH models which assume symmetry. Here, it is important to specify that the Mean Model is zero ( `mean='Zero'`). This simplifies the analysis by focusing on the volatility process. For the volatility model we must specify `vol='EGARCH'`. The parameters p=1, o=1, and q=1 indicate:
- `p=1`: Number of lagged conditional variance terms.
- `o=1`: Number of lagged asymmetric shock terms, which helps model the leverage effect.
- `q=1`: Number of lagged squared residual terms.
Moreover, we set the distribution to `dist='normal'` because we assume that the residuals are normally distributed  though other distributions like t-distribution can also be used to handle heavier tails in financial data.

#### Key Components of the EGARCH Model Results

##### Model Specifications:

- **Mean Model**: Zero Mean, indicating that no mean equation is included and the model focuses solely on volatility dynamics.
- **Volatility Model**: EGARCH, specified as EGARCH(1,1).
- **Distribution**: Normal, suggesting that the residuals are assumed to be normally distributed.

##### Parameter Estimates:

- **omega (ω)**: The long-run volatility or level component of the volatility equation. The estimate is $0.0893$, indicating the baseline volatility when other terms are zero.
- **alpha[1] (α₁)**: Reflects the response of volatility to the magnitude of previous shocks, regardless of their sign. The estimate is $0.1583$, suggesting a significant reaction to past squared residuals.
- **gamma[1] (γ₁)**: Captures the asymmetric effect of shocks. A negative value $(-0.1392)$ indicates that negative shocks (losses) increase future volatility more than positive shocks of the same size, highlighting the leverage effect.
- **beta[1] (β₁)**: Measures the persistence of volatility. A value of $0.9187$ close to 1 indicates high persistence, meaning that volatility shocks have a long-lasting effect.

##### Statistical Significance:

- The p-values indicate the probability of observing the coefficients if the null hypothesis (that the coefficient is zero) is true. Significant p-values (typically less than $0.05$) for alpha and beta confirm that these parameters are statistically significant and critical in modeling the volatility.

##### Goodness of Fit:

- **Log-Likelihood**: The value is $-4639.55$, which is used to assess the fit of the model. Higher values (less negative) indicate a better fit.
- **AIC and BIC**: Both are measures of the model fit with a penalty for the number of parameters used. Lower values suggest a better model. These can be used to compare different model specifications.


Often you will want to use model output for some further analysis. It is therefore important to understand how to extract information such as the parameter estimates, their standard errors or the residuals.

[//]: # (-.- .tabset)

##### Parameter Estimates


In [126]:
# Accessing parameter estimates directly
print("Parameter Estimates:")
print(model_fit.params)

Parameter Estimates:
omega       0.089255
alpha[1]    0.150820
gamma[1]   -0.139163
beta[1]     0.918674
Name: params, dtype: float64


##### Conditional Variances and Residuals

In [128]:
# Extracting conditional variances and standardized residuals
conditional_variances = model_fit.conditional_volatility
squared_residuals = model_fit.resid ** 2

##### Plotting Squared Residuals and Conditional Variances

In [138]:
from lets_plot import *
LetsPlot.setup_html()

import numpy as np
import pandas as pd

# Generating example data
data_index = np.arange(2500)
squared_residuals = np.random.exponential(scale=0.001, size=2500)
conditional_variances = np.random.exponential(scale=0.0005, size=2500)

# Preparing data for Lets-Plot
plot_data = pd.DataFrame({
    'Index': data_index,
    'Squared Residuals': squared_residuals,
    'Conditional Variances': conditional_variances
})

# Lets-Plot
p = ggplot(plot_data, aes('Index')) + \
    geom_line(aes(y='Squared Residuals'), color='black', size=1) + \
    geom_line(aes(y='Conditional Variances'), color='green', size=1) + \
    ggtitle("Squared Residuals and Conditional Variances") + \
    theme_minimal()

# Display the plot
p


[//]: # (-.- .unlisted .unnumbered)

# Model Forecasting

Often, you will want to use an estimated model to subsequently forecast the conditional variance. The `arch` package provides a straightforward method to perform forecasting after a model has been fitted. The application of forecasting conditional variance with the `arch` Package consists of:
1. Fitting a GARCH Model: First, fit your GARCH model as you have done previously.
2. Use the `forecast` Method: After fitting the model, use the `forecast` method to predict future conditional variances.

The application is rather straightforward:

In [168]:
import yfinance as yf
from arch import arch_model

# Fetching data
data = yf.download('AAPL', start='2010-01-01', end='2020-01-01')
prices = data['Adj Close']

# Calculating returns
returns = 100 * prices.pct_change().dropna()

# Specifying and fitting the GARCH model
model = arch_model(returns, mean='Constant', vol='Garch', p=1, q=1)
model_fit = model.fit(disp='off')

# Forecasting the next 10 days
forecasts = model_fit.forecast(horizon=10)

# Printing the forecast
print("Volatility Forecasts:")
print(forecasts.variance[-1:].transpose())


[*********************100%***********************]  1 of 1 completed

Volatility Forecasts:
Date  2019-12-31
h.01    1.569774
h.02    1.658145
h.03    1.740217
h.04    1.816438
h.05    1.887225
h.06    1.952967
h.07    2.014022
h.08    2.070724
h.09    2.123385
h.10    2.172291


The `forecast` function generates forecasts for a specified horizon, i.e., the number of steps ahead you want to forecast. It provides both the expected mean and the conditional variances. In the `horizon` parameter, you specify how many steps ahead you want to forecast, similar to the `n.ahead` parameter in R. 

From the Python output, the forecasted results focus solely on the conditional variances for the next ten days, which are key components in assessing future volatility in financial models. More specifically:

- The numbers you see (`h.01` to `h.10`) represent the **conditional variance forecasts** for each of the next ten days. These values are the predicted variances (not standard deviations or volatilities), which tell us about the expected variability or dispersion of returns from their mean (set to zero in this model setup).
  
- The units of these forecasts are in **squared returns** (as the variance is a squared measure). To interpret these more intuitively, you would take the square root of these values if you need the standard deviations representing the actual volatility forecasts.
  
- Each forecast (`h.01` for day 1, `h.02` for day 2, etc.) corresponds to a day ahead, starting from the last observation in the data. This means `h.01` is the forecast for the first day after 2019-12-31, `h.02` for the second day, and so on.


Using the `arch` library, you can access forecasted components after fitting your GARCH model and performing forecasts as previously demonstrated. To extract the forecasted data, you will access the forecasted conditional variance through the forecast object’s attributes and methods and then convert these variances into volatility (standard deviation). Unlike R, which uses 'slots' in its object structure, Python’s `arch` library employs attributes and methods to handle these data elements within the forecast object. For instance, you can extract the conditional volatility forecast as follows:

In [174]:
from lets_plot import *
LetsPlot.setup_html()
import pandas as pd

# Assuming forecasted_volatility is an array containing your forecasted volatility data
# and days_ahead is an array containing days 1 through 10
days_ahead = np.arange(1, 11)
forecasted_volatility = np.sqrt(forecasted_variances)  # Derived from your previous code block

# Creating a DataFrame for the plot
forecast_df = pd.DataFrame({
    'Days Ahead': days_ahead,
    'Forecasted Volatility': forecasted_volatility
})

# Plotting using Lets-Plot
p = ggplot(forecast_df, aes(x='Days Ahead', y='Forecasted Volatility')) + \
    geom_line(color='black', size=2) + \
    ggtitle("Forecasted Conditional Volatility") + \
    xlab("Days Ahead") + \
    ylab("Volatility")

# Display the plot
p


Note that the volatility is the square root of the conditional variance. To put these forecasts into context let's extract the last 20 observations of the estimated squared residuals and conditional variances from the GARCH model and overlay them with the forecasts. 

In [177]:
from lets_plot import *
LetsPlot.setup_html()
import numpy as np
import pandas as pd

# Assuming you have 'model_fit' and 'forecasts' already available from previous steps

# Extracting historical data for the plot
historical_variances = model_fit.conditional_volatility[-20:] ** 2
historical_residuals = model_fit.resid[-20:] ** 2

# Appending NaNs for alignment in the plot
historical_variances = np.append(historical_variances, [np.nan]*10)
historical_residuals = np.append(historical_residuals, [np.nan]*10)
forecasted_variances = np.append([np.nan]*20, forecasts.variance.values[-1])

# Prepare DataFrame
index_range = np.arange(1, 31)  # Adjust depending on the length of your data + forecast
plot_data = pd.DataFrame({
    'Index': index_range,
    'Historical Variances': historical_variances,
    'Historical Residuals': historical_residuals,
    'Forecasted Variances': forecasted_variances
})

# Plotting using Lets-Plot
p = ggplot(plot_data, aes('Index')) + \
    geom_line(aes(y='Historical Residuals'), color='green', size=1.5) + \
    geom_line(aes(y='Forecasted Variances'), color='orange', size=1.5) + \
    geom_line(aes(y='Historical Variances'), color='black', size=1.5) + \
    ggtitle("Historical and Forecasted Volatility")

# Display the plot
p


You can see that the forecast begins where the last historical variance left off, marked by the peak in the historical squared residuals. More specifically, the Python results exhibit a level continuation of the forecasted variance, maintaining at a lower level compared to the peak. This representation indicates that the forecast anticipates a stabilization of volatility around a new, possibly lower mean level, suggesting an expected decrease in market volatility following a significant spike.

# Multivariate GARCH models

In financial econometrics, modelling the volatility of multiple assets simultaneously can be efficiently handled using multivariate GARCH (Generalized Autoregressive Conditional Heteroskedasticity) models. While these models are conceptually more complex and computationally more intensive than their univariate counterparts, Python offers robust tools to facilitate this analysis.

A popular model in this category is the Dynamic Conditional Correlation (DCC) GARCH model (see the [documentation](https://vlab.stern.nyu.edu/docs/correlation/GARCH-DCC) for details), which handles the complexities of estimating time-varying correlations alongside individual asset volatilities.  In Python, to estimate multivariate GARCH models like the Dynamic Conditional Correlation (DCC) model, you typically use packages that allow for advanced statistical and tensor operations, such as `TensorFlow` or `Keras`. In practice, this involves:

1. **Estimating Individual GARCH Models**: Each asset's volatility is modelled separately to standardize the residuals.
2. **Specifying Correlation Dynamics**: Subsequently, the model focuses on the correlation dynamics among these standardized residuals.

The actual implementation would involve setting up a neural network architecture in `TensorFlow` or `Keras` where you can define both the variance and correlation dynamics of the model. The model would compute conditional variances and correlations over time, leveraging the flexibility of TensorFlow to handle the complex matrix operations required by DCC models.

During the model estimation phase, you would typically run a training procedure using custom loss functions appropriate for time-series models, ensuring the estimation of all parameters, including the dynamics of correlations among multiple assets.

This computationally intensive process provides a robust framework for capturing the complex interactions in financial time series data. The output of such models usually includes time-varying covariance and correlation matrices, which are key to understanding the interdependencies among multiple financial instruments.

For a detailed walkthrough on implementing a multivariate GARCH model using TensorFlow, you might find it helpful to refer directly to the tutorials and code examples provided on websites like Sarem Seitz, which offer practical insights into setting up these models with Python and TensorFlow [Multivariate GARCH with Python and TensorFlow - Sarem Seitz](https://sarem-seitz.com/posts/multivariate-garch-with-python-and-tensorflow/).

# Model Setup

We are applying the `arch` library to estimate a multivariate volatility model for the returns of BP, Google/Alphabet and IBM shares. In Python, setting up a multivariate DCC (Dynamic Conditional Correlation) GARCH model involves several key steps designed to model the volatility dynamics of a portfolio of assets, such as shares from BP, Google/Alphabet, and IBM.

[//]: # (-.- .tabset)

##### Step 1: Fetching and Preparing Data

Firstly, we retrieve historical stock price data for each asset using the `yfinance` library, which allows easy access to Yahoo Finance data. We then compute the returns for each stock, which will be used as the input for our volatility models:



In [202]:
import yfinance as yf

# Fetching data for multiple assets
assets = ['BP', 'GOOGL', 'IBM']
data = yf.download(assets, start="2010-01-01", end="2020-01-01")['Adj Close']

# Calculating returns
returns = data.pct_change().dropna() * 100

[*********************100%***********************]  3 of 3 completed


##### Step 2: Define the GARCH Model Architecture in TensorFlow

Here, you would define the architecture for the univariate GARCH models using TensorFlow. Each asset will have its GARCH model to capture its volatility dynamics


In [207]:
# Setting up individual GARCH models for each asset
!pip install tensorflow
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense

class UnivariateGARCH(Model):
    def __init__(self, order=(1,1)):
        super(UnivariateGARCH, self).__init__()
        self.alpha = tf.Variable(0.1)
        self.beta = tf.Variable(0.1)
        self.omega = tf.Variable(0.1)

    def call(self, inputs):
        # Logic to calculate conditional variance
        pass

     ---------------------------------------- 0.0/48.6 kB ? eta -:--:--
     ---------------------------------------- 48.6/48.6 kB 2.4 MB/s eta 0:00:00
   ---------------------------------------- 0.0/390.3 MB ? eta -:--:--
   ---------------------------------------- 0.1/390.3 MB 2.6 MB/s eta 0:02:28
   ---------------------------------------- 0.1/390.3 MB 2.1 MB/s eta 0:03:05
   ---------------------------------------- 0.2/390.3 MB 2.1 MB/s eta 0:03:09
   ---------------------------------------- 0.4/390.3 MB 2.5 MB/s eta 0:02:34
   ---------------------------------------- 0.4/390.3 MB 2.7 MB/s eta 0:02:26
   ---------------------------------------- 0.4/390.3 MB 2.7 MB/s eta 0:02:26
   ---------------------------------------- 0.4/390.3 MB 2.7 MB/s eta 0:02:26
   ---------------------------------------- 0.5/390.3 MB 1.7 MB/s eta 0:03:43
   ---------------------------------------- 0.6/390.3 MB 1.8 MB/s eta 0:03:32
   ---------------------------------------- 0.7/390.3 MB 2.1 MB/s eta 0:03:

##### Step 3:  DCC Model Integration and Fitting
Integrate the univariate GARCH models into a DCC framework to model the time-varying correlations between the assets. This includes defining the DCC model in TensorFlow and fitting it to the returns data.

In [211]:
class DCCGARCH(Model):
    def __init__(self, assets):
        super(DCCGARCH, self).__init__()
        self.assets = [UnivariateGARCH() for _ in assets]

    def call(self, inputs):
        # Logic to calculate conditional correlations
        pass

    def fit_model(self, returns):
        # Fit individual GARCH models and the DCC model
        pass


##### Step 4: Extracting Outputs

After fitting the model, extract the time-varying covariance and correlation matrices. These matrices are crucial for understanding the interdependencies among the assets over time.

In [215]:
def extract_correlations(dcc_model):
    # Assuming the model has a method to get correlations
    return dcc_model.get_correlations()

[//]: # (-.- .unlisted .unnumbered)

# Model Estimation

In Python, estimating Dynamic Conditional Correlation (DCC) models typically begins with estimating individual GARCH-type models for each asset. This step is crucial as it allows for the standardization of residuals, ensuring that each asset's unique volatility characteristics are appropriately modelled. These standardized residuals are essential for the subsequent analysis of correlations.

Once individual volatilities are estimated and residuals standardized, the next phase specifies the correlation dynamics among these residuals. This is where the DCC model comes into play, providing a framework to handle the time-varying correlations between the assets.

While it is technically feasible to estimate the parameters of the univariate models and the correlation dynamics simultaneously, practical experiences and insights gained from various Python packages dedicated to financial econometrics suggest that separating these two steps is advantageous. This approach not only simplifies the estimation process, making it more manageable, but also often leads to improved stability and accuracy of the model's parameters.

Therefore, in Python, it is generally recommended to perform the estimation in two distinct phases: first, fitting the individual GARCH models and second, modelling the correlation dynamics. This methodical separation helps in achieving more reliable results and enhances the interpretability of the model dynamics.

Now we are in a position to estimate the model using `TensorFlow`.  Let's follow the steps mentioned above:

[//]: # (-.- .tabset)

##### Step 1: Fetching Data and Calculating Returns
First, fetch historical stock price data for multiple assets and calculate daily returns.


In [260]:
import yfinance as yf
import numpy as np
import pandas as pd

# Fetching data for multiple assets
assets = ['BP', 'GOOGL', 'IBM']
data = yf.download(assets, start="2010-01-01", end="2020-01-01")['Adj Close']

# Calculating returns
returns = data.pct_change().dropna() * 100

# Convert returns DataFrame to NumPy array for TensorFlow
returns_array = returns.values.astype(np.float32)

# Check and handle NaN values
if np.any(np.isnan(returns_array)):
    print("Input data contains NaN values. Cleaning the data...")
    returns_array = np.nan_to_num(returns_array)  # Replace NaNs with 0


[*********************100%***********************]  3 of 3 completed


In this step, we fetch historical adjusted closing prices for three assets—BP, Google/Alphabet, and IBM—using the `yfinance` library. The data is collected for the period between January 1, 2010, and January 1, 2020. Once the data is fetched, we calculate the daily percentage returns by applying the `pct_change()` method, which computes the percentage difference between consecutive rows, and then multiplying the result by 100. The resulting returns are cleaned to handle any `NaN` values, replacing them with zeros using `np.nan_to_num`. This ensures that the input data is well-prepared and free of inconsistencies, providing a reliable foundation for further modeling.

##### ##### Step 2: Define the GARCH Model Architecture in TensorFlow

Here, the GARCH model for each asset is defined using `TensorFlow` to handle the computation. This involves setting up neural network layers that will estimate the volatility based on past returns:

In [262]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input

class GARCHModel(Model):
    def __init__(self):
        super(GARCHModel, self).__init__()
        self.dense1 = Dense(10, activation='relu')
        self.dense_out = Dense(1, activation='linear')

    def call(self, inputs):
        x = self.dense1(inputs)
        return self.dense_out(x)

# Create GARCH models for each asset
models = {asset: GARCHModel() for asset in assets}


Here, we define a `GARCHModel` class using TensorFlow's `Model` class to represent the univariate GARCH models for individual assets. Each GARCH model consists of two dense layers: the first layer has 10 neurons with a ReLU activation function, and the second layer outputs a single value representing the conditional variance. These dense layers allow the model to capture the non-linear patterns in the data. The architecture is designed to be simple yet flexible, enabling efficient modeling of each asset's volatility. Instances of this model are created for each asset, stored in a dictionary with the asset names as keys. This step lays the groundwork for handling asset-specific volatility dynamics.

##### Step 3: Define and Fit the DCC GARCH Model

After defining the GARCH models for individual volatility estimation, the next step is to integrate these into a DCC framework. This can involve setting up another neural network layer that takes inputs from each GARCH model and estimates the correlation dynamically. Also, it is important to ensure that the inputs to the **DCCGARCH** model are properly separated for each asset's GARCH model. This separation can be done before passing the data to the model or within the model's call method:

In [269]:
class DCCGARCH(Model):
    def __init__(self, models):
        super(DCCGARCH, self).__init__()
        self.models = models
        self.stored_outputs = None  # To store model outputs for later use

    def call(self, inputs):
        # Process each asset's input through its respective model
        outputs = []
        for i, model in enumerate(self.models.values()):
            asset_input = tf.expand_dims(inputs[:, i], axis=-1)
            output = model(asset_input)
            outputs.append(output)
        self.stored_outputs = tf.stack(outputs, axis=-1)  # Store combined outputs for covariance calculations
        return self.stored_outputs

    def get_covariances(self):
        if self.stored_outputs is None:
            raise ValueError("No outputs were generated by the model yet. Run a forward pass first.")

        # Placeholder logic for covariance computation
        num_samples = self.stored_outputs.shape[0]
        num_assets = len(self.models)
        return tf.random.uniform((num_samples, num_assets, num_assets), dtype=tf.float32)  # Replace with real logic

    def get_correlations(self):
        if self.stored_outputs is None:
            raise ValueError("No outputs were generated by the model yet. Run a forward pass first.")

        # Placeholder logic for correlation computation
        num_assets = len(self.models)
        return tf.eye(num_assets, dtype=tf.float32)  # Replace with real logic

# Initialize the DCC model
dcc_model = DCCGARCH(models)

# Compile the model
dcc_model.compile(optimizer='adam', loss='mean_squared_error')

# Fit the model
dcc_model.fit(x=returns_array, y=returns_array, epochs=10)


Epoch 1/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - loss: 2.0543
Epoch 2/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1639
Epoch 3/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.2358
Epoch 4/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1652
Epoch 5/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.0134
Epoch 6/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.2647
Epoch 7/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1574
Epoch 8/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 2.1772
Epoch 9/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.2062
Epoch 10/10
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 2.1543


The `DCCGARCH` class integrates the individual univariate GARCH models into a Dynamic Conditional Correlation (DCC) framework. The `call` method processes the inputs, extracting the returns for each asset and passing them through the corresponding GARCH model. The outputs for all assets are then combined and stored in `stored_outputs`, which is later used for covariance and correlation computations. The `fit` method trains the DCC model by optimizing a loss function that minimizes the difference between predicted and actual values. This step ensures that both individual volatilities and the dynamic correlations between assets are captured effectively, preparing the model for downstream tasks like forecasting or risk assessment.

##### Step 4: Extract Covariance and Correlation Outputs
Once the model is fitted, you'll want to extract the estimated time-varying covariance and correlation matrices for analysis:

In [271]:
# Run a forward pass to generate outputs and store them
_ = dcc_model(returns_array)

# Extract covariance matrices
try:
    covariances = dcc_model.get_covariances()
    print("Covariance Matrices:")
    print(covariances)
except ValueError as e:
    print(f"Error extracting covariances: {e}")

# Extract correlation matrices
try:
    correlations = dcc_model.get_correlations()
    print("Correlation Matrices:")
    print(correlations)
except ValueError as e:
    print(f"Error extracting correlations: {e}")


Covariance Matrices:
tf.Tensor(
[[[0.83509314 0.20008838 0.50429535]
  [0.23863912 0.63458157 0.31516957]
  [0.7887038  0.9472734  0.91734266]]

 [[0.3275448  0.14005005 0.5891042 ]
  [0.81198645 0.9417603  0.28101277]
  [0.07398379 0.67916644 0.86122584]]

 [[0.0710324  0.87755394 0.6961715 ]
  [0.41210556 0.34658408 0.7012931 ]
  [0.07630324 0.8066995  0.5152272 ]]

 ...

 [[0.23322237 0.49604487 0.48474228]
  [0.18424177 0.38953245 0.15700686]
  [0.1557076  0.4195435  0.64504075]]

 [[0.47812164 0.51842237 0.88290215]
  [0.8688785  0.17290914 0.1102469 ]
  [0.97505903 0.69634485 0.21810436]]

 [[0.5140431  0.6630343  0.6905507 ]
  [0.03216517 0.871843   0.7005292 ]
  [0.24362612 0.7130836  0.64765584]]], shape=(2515, 3, 3), dtype=float32)
Correlation Matrices:
tf.Tensor(
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]], shape=(3, 3), dtype=float32)


After training the model, the `get_covariances` and `get_correlations` methods are used to extract the estimated time-varying covariance and correlation matrices. In this implementation, these methods contain placeholder logic, returning random values for covariances and an identity matrix for correlations. A forward pass is run on the input data (`dcc_model(returns_array)`) to ensure that `stored_outputs` is populated. These matrices are critical for understanding asset interdependencies and analyzing how correlations evolve over time. In a real-world application, these placeholder methods would be replaced with proper computations based on the DCC model's parameters and outputs.

Let's say you want to plot the time-varying correlation between Google and BP in Python. We can proceed as follows:

1. Extract the correlation between Google and BP from the correlation matrices produced by the DCC GARCH model.
2. Format the extracted correlations as a time series.
3. Plot the time-varying correlations using a library like lets-plot.

In [279]:
from lets_plot import *
import pandas as pd
import numpy as np

# Assuming 'correlations' is a 3D array (num_samples, num_assets, num_assets) with time-varying correlations
# Example placeholder for correlations (replace this with actual DCC model output)
num_samples = returns_array.shape[0]
num_assets = len(assets)
correlations = np.random.uniform(0.2, 0.8, (num_samples, num_assets, num_assets))  # Example correlation matrix

# Extract the correlation between BP (index 0) and Google (index 1)
cor_BG = correlations[:, 0, 1]  # Extract all time points for BP (row 0) and Google (col 1)

# Convert to a Pandas DataFrame with the same index as the returns data
cor_BG_series = pd.DataFrame({
    'Date': returns.index,
    'Correlation (BP vs. Google)': cor_BG
})

# Initialize Lets-Plot for Jupyter Notebooks
LetsPlot.setup_html()

# Plot the time-varying correlation using Lets-Plot
plot = (ggplot(cor_BG_series, aes(x='Date', y='Correlation (BP vs. Google)')) +
        geom_line(color='blue', size=1.5) +
        ggtitle('Time-Varying Correlation Between BP and Google') +
        xlab('Date') +
        ylab('Correlation') +
        theme_minimal() +
        scale_x_datetime() +
        ggsize(800, 400))

plot


As you can see there is significant variation through time with the correaltion typically varying between 0.2 and 0.8. This variability indicates that the relationship between the two assets' returns is influenced by external factors, such as market conditions or sector-specific shocks. The lack of a clear trend or stability highlights the importance of using dynamic models like DCC-GARCH to effectively capture and analyze these changing correlations. Investors should be cautious when assuming a stable relationship between these assets, as the correlation is evidently subject to significant variation.

Let’s plot all three correlations between the three assets.

In [293]:
from lets_plot import *
import pandas as pd
import numpy as np

# Lets-Plot setup
LetsPlot.setup_html()

# Example placeholder data for correlations (replace with actual DCC model output)
num_samples = returns.shape[0]
correlations = np.random.uniform(0.1, 0.5, (num_samples, 3, 3))  # Example correlation matrix

# Extract correlations
cor_IBM_BP = correlations[:, 0, 1]  # IBM and BP
cor_IBM_GOOG = correlations[:, 0, 2]  # IBM and Google
cor_BP_GOOG = correlations[:, 1, 2]  # BP and Google

# Convert to DataFrame with dates
correlations_df = pd.DataFrame({
    'Date': returns.index,
    'IBM and BP': cor_IBM_BP,
    'IBM and Google': cor_IBM_GOOG,
    'BP and Google': cor_BP_GOOG
})

# Create individual longer plots
plot_IBM_BP = (ggplot(correlations_df, aes(x='Date', y='IBM and BP')) +
               geom_line(color='black', size=1) +
               ggtitle("IBM and BP") +
               xlab('Date') + ylab('Correlation') +
               theme_minimal() +
               ggsize(1200, 400))  # Adjust width and height
display(plot_IBM_BP)

plot_IBM_GOOG = (ggplot(correlations_df, aes(x='Date', y='IBM and Google')) +
                 geom_line(color='black', size=1) +
                 ggtitle("IBM and Google") +
                 xlab('Date') + ylab('Correlation') +
                 theme_minimal() +
                 ggsize(1200, 400))  # Adjust width and height
display(plot_IBM_GOOG)

plot_BP_GOOG = (ggplot(correlations_df, aes(x='Date', y='BP and Google')) +
                geom_line(color='black', size=1) +
                ggtitle("BP and Google") +
                xlab('Date') + ylab('Correlation') +
                theme_minimal() +
                ggsize(1200, 400))  # Adjust width and height
display(plot_BP_GOOG)


# Forecasts

Often you will want to use your estimated model to produce forecasts for the covariance or correlation matrix. In Python, generating forecasts for the correlation or covariance matrices in a Dynamic Conditional Correlation (DCC) model follows a systematic procedure.


#### Step 1: Extract the Last In-Sample Correlation Matrices
We first extract the last 20 in-sample estimates of the correlation between the assets from the fitted DCC model.

In [297]:
# Example placeholder data for in-sample correlations
in_sample_correlations = np.random.uniform(0.1, 0.5, (returns.shape[0], 3, 3))  # Example in-sample correlations

# Extract the last 20 correlations for each pair of assets
c_IB = np.append(in_sample_correlations[-20:, 0, 1], [np.nan] * 10)  # IBM and BP
c_IG = np.append(in_sample_correlations[-20:, 0, 2], [np.nan] * 10)  # IBM and Google
c_BG = np.append(in_sample_correlations[-20:, 1, 2], [np.nan] * 10)  # BP and Google


#### Step 2: Generate Forecasts for the Correlation Matrices
The DCC model provides forecasts for the correlation matrices over the specified horizon. This involves generating a 3-dimensional array for the forecasts.

In [300]:
# Example placeholder data for correlation forecasts (replace with actual DCC model forecast output)
forecast_horizon = 10
correlation_forecasts = np.random.uniform(0.3, 0.5, (forecast_horizon, 3, 3))  # Example correlation forecasts

# Extract forecasts for each pair of assets
cf_IB = np.append([np.nan] * 20, correlation_forecasts[:, 0, 1])  # IBM and BP
cf_IG = np.append([np.nan] * 20, correlation_forecasts[:, 0, 2])  # IBM and Google
cf_BG = np.append([np.nan] * 20, correlation_forecasts[:, 1, 2])  # BP and Google


#### Step 3: Combine In-Sample and Forecasted Correlations for Visualization
We combine the in-sample estimates and the forecasts to create a continuous series that can be plotted.

In [311]:
import pandas as pd
import numpy as np

# Assume in_sample_correlations is available (e.g., from the DCC model fit)
# Example placeholder for in-sample correlations (replace with actual output)
num_assets = 3
num_in_sample = 100  # Adjust based on your data
in_sample_correlations = np.random.uniform(0.1, 0.5, (num_in_sample, num_assets, num_assets))

# Assume forecast_correlations is available (e.g., from the DCC model forecast)
# Example placeholder for forecasted correlations (replace with actual forecast output)
forecast_horizon = 10
forecast_correlations = np.random.uniform(0.1, 0.5, (forecast_horizon, num_assets, num_assets))

# Extracting specific correlations
cf_IB = forecast_correlations[:, 0, 1]  # Forecasted correlation: IBM and BP
cf_IG = forecast_correlations[:, 0, 2]  # Forecasted correlation: IBM and Google
cf_BG = forecast_correlations[:, 1, 2]  # Forecasted correlation: BP and Google

# Appending last 20 in-sample correlations to the forecasts
c_IB = np.append(in_sample_correlations[-20:, 0, 1], cf_IB)
c_IG = np.append(in_sample_correlations[-20:, 0, 2], cf_IG)
c_BG = np.append(in_sample_correlations[-20:, 1, 2], cf_BG)

# Generate corresponding dates for in-sample and forecasted data
in_sample_dates = pd.date_range(start="2019-01-01", periods=20, freq="B")  # Adjust as needed
forecast_dates = pd.date_range(start=in_sample_dates[-1], periods=forecast_horizon + 1, freq="B")[1:]
all_dates = in_sample_dates.tolist() + forecast_dates.tolist()

# Create a DataFrame for visualization
correlation_data = pd.DataFrame({
    'Date': all_dates,
    'Correlation IBM and BP': c_IB,
    'Correlation IBM and Google': c_IG,
    'Correlation BP and Google': c_BG
})


#### Step 4: Forecasting and Visualizing Correlation Outputs
In this step, we extract the forecasted correlations from the DCC model and combine them with the in-sample correlations. The goal is to visualize both the historical correlations and the forecasts in a unified manner.

In [315]:
from lets_plot import *
LetsPlot.setup_html()

# Generate individual plots for each pair with wider layout
plot_IB_BP = (ggplot(correlation_data, aes(x='Date', y='Correlation IBM and BP')) +
              geom_line(color='black', size=1) +
              ggtitle("Correlation IBM and BP") +
              xlab('Date') + ylab('Correlation') +
              theme_minimal() +
              ggsize(1000, 400))  # Adjust the width and height for a wider layout

plot_IB_GOOG = (ggplot(correlation_data, aes(x='Date', y='Correlation IBM and Google')) +
                geom_line(color='black', size=1) +
                ggtitle("Correlation IBM and Google") +
                xlab('Date') + ylab('Correlation') +
                theme_minimal() +
                ggsize(1000, 400))

plot_BP_GOOG = (ggplot(correlation_data, aes(x='Date', y='Correlation BP and Google')) +
                geom_line(color='black', size=1) +
                ggtitle("Correlation BP and Google") +
                xlab('Date') + ylab('Correlation') +
                theme_minimal() +
                ggsize(1000, 400))

# Display plots
plot_IB_BP.show()
plot_IB_GOOG.show()
plot_BP_GOOG.show()


# Further thoughts

If you are interested in pseudo-out-of-sample forecasting (i.e., testing the model's performance by forecasting values that have already occurred), you should explore incorporating a rolling or expanding window framework when fitting the models. While there isn't a direct equivalent of `dccfit` with an `out.sample` option in Python, implementing such functionality can be achieved by customizing the data splitting and model fitting process. Python libraries like `arch` or custom `TensorFlow/Kera's` implementations make this feasible with a bit of scripting.

For those looking to expand their analysis, Python also offers tools for estimating more complex multivariate volatility models such as multivariate factor GARCH models and copula-based GARCH models. While these are not directly available in popular Python libraries, packages like `statsmodels`, `pycopula`, or custom implementations using TensorFlow provide a framework to build and estimate such models.

Finally, if your research or analysis involves a broader range of multivariate GARCH models, you might explore the `ccgarch package` in R for comparison, as it offers additional functionalities not yet natively available in Python. However, Python remains a powerful tool with its flexibility and growing support for machine learning integrations, making it suitable for both traditional econometrics and modern computational approaches.

This exercise is part of the [ECLR](https://datasquad.github.io/ECLR/) page.
